# NN in PyTorch: Beginner’s Walkthrough

Text Classification problem with NN.

Setting Up the Environment

In [1]:
# verify the GPU setup by running this code

import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU Available: False
GPU Name: None


In [2]:
# Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

## Data Preparation

From Hugging Face’s datasets library:

 
Dataset: AG News (available in PyTorch’s torchtext library)

Description: Contains 120,000 training samples and 7,600 test samples of news headlines and short descriptions, along with their corresponding labels.

Input: Text data (news headlines and descriptions)

Output: A category label (1 to 4)

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Load the AG News dataset
dataset = load_dataset("ag_news")

c:\Users\vika.dubanevica\Downloads\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\vika.dubanevica\Downloads\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vika.dubanevica\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activa

In [4]:
# Load a tokenizer (e.g., BERT tokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the dataset
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Format the dataset for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

c:\Users\vika.dubanevica\Downloads\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vika.dubanevica\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 7600/7600 [00:03<00:00, 2301.60 examples/s]


In [5]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_dataloader = DataLoader(tokenized_datasets["train"], batch_size=32, shuffle=True)
test_dataloader = DataLoader(tokenized_datasets["test"], batch_size=32)

## Model Building

In [6]:
import torch.nn as nn

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, 128)  # First hidden layer
        self.relu = nn.ReLU()  # Activation function
        self.fc2 = nn.Linear(128, num_classes)  # Output layer
    
    def forward(self, input_ids):
        # Embed the input tokens
        embedded = self.embedding(input_ids).mean(dim=1)  # Mean pooling
        # Pass through the network
        x = self.fc1(embedded)
        x = self.relu(x)
        output = self.fc2(x)
        return output

# Initialize the model
vocab_size = tokenizer.vocab_size
embed_dim = 128
num_classes = 4
model = TextClassifier(vocab_size, embed_dim, num_classes)

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

## Training

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_dataloader:
        # Move data to device
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader)}")

Epoch 1, Loss: 0.631220155475537
Epoch 2, Loss: 0.302766460921367
Epoch 3, Loss: 0.24938737150728701


## Evaluation

In [9]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9046


In [ ]:
def classify_text(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(inputs["input_ids"])
    prediction = torch.argmax(outputs, dim=1).item()
    categories = ["World", "Sports", "Business", "Science/Technology"]
    return categories[prediction]

# Example usage
text = "French veteran Gael Monfils becomes the oldest player to win an ATP Tour singles title."
print(f"Predicted Category: {classify_text(text)}")


Predicted Category: Sports
